# Feature Engineering: PowerCo SME Customer Churn

**Goal:** turn the cleaned client data (`clean_data_after_eda.csv`, output of the EDA stage) into model-ready features that capture tenure, consumption behaviour, margins and price sensitivity.

**Steps:** 1. Load data, 2. Remove redundant columns, 3. Date features, 4. Consumption features, 5. Margin features, 6. Price-sensitivity features, 7. Encode categoricals, 8. Summary and save.

## 1. Load data

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('./clean_data_after_eda.csv')

# Convert date columns
date_cols = ['date_activ', 'date_end', 'date_modif_prod', 'date_renewal']
for col in date_cols:
    df[col] = pd.to_datetime(df[col], format='%Y-%m-%d')

print(f"Original shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

Original shape: (14606, 44)
Columns: ['id', 'channel_sales', 'cons_12m', 'cons_gas_12m', 'cons_last_month', 'date_activ', 'date_end', 'date_modif_prod', 'date_renewal', 'forecast_cons_12m', 'forecast_cons_year', 'forecast_discount_energy', 'forecast_meter_rent_12m', 'forecast_price_energy_off_peak', 'forecast_price_energy_peak', 'forecast_price_pow_off_peak', 'has_gas', 'imp_cons', 'margin_gross_pow_ele', 'margin_net_pow_ele', 'nb_prod_act', 'net_margin', 'num_years_antig', 'origin_up', 'pow_max', 'var_year_price_off_peak_var', 'var_year_price_peak_var', 'var_year_price_mid_peak_var', 'var_year_price_off_peak_fix', 'var_year_price_peak_fix', 'var_year_price_mid_peak_fix', 'var_year_price_off_peak', 'var_year_price_peak', 'var_year_price_mid_peak', 'var_6m_price_off_peak_var', 'var_6m_price_peak_var', 'var_6m_price_mid_peak_var', 'var_6m_price_off_peak_fix', 'var_6m_price_peak_fix', 'var_6m_price_mid_peak_fix', 'var_6m_price_off_peak', 'var_6m_price_peak', 'var_6m_price_mid_peak', 'chur

## 2. Remove redundant columns

`margin_gross_pow_ele` and `margin_net_pow_ele` are almost perfectly correlated, and `forecast_cons_year` duplicates `forecast_cons_12m`. Keeping both of each pair adds noise without information, so one of each is dropped.

In [2]:
# margin_gross_pow_ele and margin_net_pow_ele are nearly identical
print("Correlation between margin_gross_pow_ele and margin_net_pow_ele:")
print(df[['margin_gross_pow_ele', 'margin_net_pow_ele']].corr())
df = df.drop(columns=['margin_net_pow_ele'])

# forecast_cons_year and forecast_cons_12m are similar consumption forecasts
# Keep forecast_cons_12m as it is more granular
df = df.drop(columns=['forecast_cons_year'])

print(f"\nShape after removing redundant columns: {df.shape}")

Correlation between margin_gross_pow_ele and margin_net_pow_ele:
                      margin_gross_pow_ele  margin_net_pow_ele
margin_gross_pow_ele              1.000000            0.999914
margin_net_pow_ele                0.999914            1.000000

Shape after removing redundant columns: (14606, 42)


## 3. Date features

Raw dates are hard for a model to use, so they are converted into durations (in months) relative to a single reference date: how long the customer has been with PowerCo, how long until the contract ends, how long since the product was last changed, and how long since the last renewal.

In [3]:
# Reference date ("today") for all duration features.
# ASSUMPTION: the data has no explicit snapshot date, so we use 2016-01-01,
# the end of the 2015 price history (all contracts end on or after Jan 2016).
reference_date = pd.Timestamp('2016-01-01')

# Months since activation (customer tenure in months)
df['months_activ'] = ((reference_date - df['date_activ']).dt.days / 30).astype(int)

# Months until contract end (urgency / renewal pressure)
df['months_to_end'] = ((df['date_end'] - reference_date).dt.days / 30).astype(int)

# Months since last product modification (recency of product change)
df['months_since_modif'] = ((reference_date - df['date_modif_prod']).dt.days / 30).astype(int)

# Months since the last contract renewal (renewal dates are in the past, so reference - date)
df['months_since_renewal'] = ((reference_date - df['date_renewal']).dt.days / 30).astype(int)

# Has the product ever been modified? (date_modif_prod != date_activ signals a change)
df['has_modified_product'] = (df['date_modif_prod'] != df['date_activ']).astype(int)

# Drop the raw date columns, the engineered versions are more model-friendly
df = df.drop(columns=date_cols)

print(f"Shape after date feature engineering: {df.shape}")
print("New date features:", ['months_activ', 'months_to_end', 'months_since_modif',
                             'months_since_renewal', 'has_modified_product'])

Shape after date feature engineering: (14606, 43)
New date features: ['months_activ', 'months_to_end', 'months_since_modif', 'months_since_renewal', 'has_modified_product']


## 4. Consumption features

These describe *how* a customer uses energy, not just how much: recent usage vs. their yearly average, total electricity plus gas, the share that is gas, and how far the forecast is from actual use.

In [4]:
# Ratio: last month's consumption vs 12-month average
# Captures whether usage is trending up or down recently
df['cons_last_month_vs_12m_ratio'] = df['cons_last_month'] / (df['cons_12m'] / 12 + 1e-6)

# Total energy consumption (electricity + gas combined)
df['total_cons_12m'] = df['cons_12m'] + df['cons_gas_12m']

# Gas as a proportion of total consumption (0 = electricity only, 1 = gas only)
df['gas_share_of_total'] = df['cons_gas_12m'] / (df['total_cons_12m'] + 1e-6)

# Forecast vs actual consumption gap (over-/under-forecasting)
df['forecast_vs_actual_cons'] = df['forecast_cons_12m'] - df['cons_12m']

print(f"Shape after consumption features: {df.shape}")

Shape after consumption features: (14606, 47)


## 5. Margin and financial features

Profitability per unit of use and per kW of capacity, plus a flag for customers who receive a discount.

In [5]:
# Net margin per unit of consumption (profitability per kWh)
df['net_margin_per_cons'] = df['net_margin'] / (df['cons_12m'] + 1e-6)

# Margin relative to power capacity (margin per kW of max power)
df['margin_per_pow_max'] = df['margin_gross_pow_ele'] / (df['pow_max'] + 1e-6)

# Discount flag: is the customer receiving any discount?
df['has_discount'] = (df['forecast_discount_energy'] > 0).astype(int)

print(f"Shape after margin features: {df.shape}")

Shape after margin features: (14606, 50)


## 6. Price-sensitivity features

Used to test the hypothesis that price sensitivity drives churn. Only like-for-like quantities are combined: energy prices with energy prices, and price *changes* that already include both the variable and fixed part.

In [6]:
# Peak-to-offpeak ratio: how much more expensive is peak vs off-peak energy?
df['peak_offpeak_ratio'] = df['forecast_price_energy_peak'] / (df['forecast_price_energy_off_peak'] + 1e-6)

# Total annual price exposure: consumption * forecasted off-peak energy price
df['price_energy_exposure'] = df['cons_12m'] * df['forecast_price_energy_off_peak']

# Price change acceleration: is the rate of change increasing?
# var_year_price_off_peak and var_6m_price_off_peak already combine the variable
# and fixed off-peak price changes, so they are used directly.
# Positive = prices rising faster recently; negative = slowing down
df['price_change_acceleration'] = df['var_6m_price_off_peak'] - (df['var_year_price_off_peak'] / 2)

print(f"Shape after price sensitivity features: {df.shape}")

Shape after price sensitivity features: (14606, 53)


## 7. Encode categorical columns

`has_gas` becomes 0/1. `channel_sales` and `origin_up` are one-hot encoded; the `'MISSING'` placeholder is turned into NaN first so it gets its own indicator column instead of being treated as a real category.

In [7]:
# has_gas: 't'/'f' -> 1/0
df['has_gas'] = (df['has_gas'] == 't').astype(int)

# channel_sales and origin_up: one-hot encode
# Replace 'MISSING' with NaN first so it gets its own indicator column
df['channel_sales'] = df['channel_sales'].replace('MISSING', np.nan)
df = pd.get_dummies(df, columns=['channel_sales', 'origin_up'], dummy_na=True, prefix=['channel', 'origin'])

# Convert boolean dummy columns to int
bool_cols = df.select_dtypes(include='bool').columns
df[bool_cols] = df[bool_cols].astype(int)

print(f"Shape after encoding categoricals: {df.shape}")

Shape after encoding categoricals: (14606, 66)


## 8. Summary and save

In [8]:
print("=== Final Dataset Summary ===")
print(f"Shape: {df.shape}")
print(f"\nAll columns ({len(df.columns)}):")
for col in df.columns:
    print(f"  {col}: {df[col].dtype}")

print(f"\nMissing values: {df.isnull().sum().sum()}")
print(f"Churn rate: {df['churn'].mean()*100:.1f}%")

=== Final Dataset Summary ===
Shape: (14606, 66)

All columns (66):
  id: str
  cons_12m: int64
  cons_gas_12m: int64
  cons_last_month: int64
  forecast_cons_12m: float64
  forecast_discount_energy: float64
  forecast_meter_rent_12m: float64
  forecast_price_energy_off_peak: float64
  forecast_price_energy_peak: float64
  forecast_price_pow_off_peak: float64
  has_gas: int64
  imp_cons: float64
  margin_gross_pow_ele: float64
  nb_prod_act: int64
  net_margin: float64
  num_years_antig: int64
  pow_max: float64
  var_year_price_off_peak_var: float64
  var_year_price_peak_var: float64
  var_year_price_mid_peak_var: float64
  var_year_price_off_peak_fix: float64
  var_year_price_peak_fix: float64
  var_year_price_mid_peak_fix: float64
  var_year_price_off_peak: float64
  var_year_price_peak: float64
  var_year_price_mid_peak: float64
  var_6m_price_off_peak_var: float64
  var_6m_price_peak_var: float64
  var_6m_price_mid_peak_var: float64
  var_6m_price_off_peak_fix: float64
  var_6m_pr

In [9]:
# Save the engineered dataset for the modelling stage
df.to_csv('feature_engineered_data.csv', index=False)
print(f"Saved feature_engineered_data.csv {df.shape}")

Saved feature_engineered_data.csv (14606, 66)
